# Schema Mapping — Raw → Asclepius v1

**Purpose:** Map raw PBMC 3k metadata fields onto the Asclepius Biological State Graph v1 schema.

See `docs/schema_v1.md` for the target schema specification.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
from datetime import date
from asclepius.schema import Experiment, Sample, CellState, ProcessingPipeline
from asclepius.models import BiologicalStateGraph

## 1 — Field Mapping Table

| Raw field | Asclepius entity | Target field | Transformation |
|---|---|---|---|
| `barcode` | CellState | `id` | direct |
| `louvain` (cluster) | CellState | `annotation_label` | direct |
| GEO accession | Experiment | `id` | manual entry |
| `"human"` | Experiment | `organism` | → `NCBITaxon:9606` |
| `"10x_v2"` | Experiment | `assay_type` | → `EFO:0009899` |
| pipeline git SHA | ProcessingPipeline | `software_version` | manual entry |

In [ ]:
# ── Build an Experiment object from PBMC 3k metadata ─────────────────────────
experiment = Experiment(
    id='GSE96315',
    organism='NCBITaxon:9606',
    assay_type='EFO:0009899',
    date=date(2017, 1, 1),
    lab='10x Genomics',
    pipeline_version='1.0.0',
)
print(experiment)

In [ ]:
# ── Build a ProcessingPipeline ────────────────────────────────────────────────
pipeline = ProcessingPipeline(
    id='cellranger_v2',
    reference_genome='GRCh38',
    normalization_method='log1p_CP10K',
    batch_correction_method='none',
    software_version='2.0.0',
)
print(pipeline)

In [ ]:
# ── Assemble the state graph ──────────────────────────────────────────────────
graph = BiologicalStateGraph(experiment=experiment, pipeline=pipeline)

sample = Sample(
    id='PBMC3K_S1',
    experiment_id='GSE96315',
    perturbation_type='none',
)
graph.add_sample(sample)

# Example cell state (would normally be created from the loaded matrix)
cell = CellState(
    id='AAACATACAACCAC-1',
    sample_id='PBMC3K_S1',
    annotation_label='CD4 T cells',
    processing_version='1.0.0',
)
graph.add_cell_state(cell)

print(graph.summary())

## 2 — Gaps Identified

Fields required by the schema that are **absent** from the raw PBMC 3k dataset:

| Missing field | Schema entity | Recommended source |
|---|---|---|
| `tissue_ontology_term_id` | CellState | UBERON:0000178 (blood) — add manually |
| `disease_ontology_term_id` | CellState | PATO:0000461 (normal) — add manually |
| `suspension_type` | CellState | `"cell"` — infer from protocol |
| `donor_id` | CellState | Not released publicly — mark as unknown |